# Pilot — Part B (P5 reader · P6 judge · P7 N_gen + cost · P8 lock files)
**Settings:** Accelerator **GPU T4 x2** (never P100) · Internet **On** · no secrets needed (all models are ungated).
Run top to bottom. **If a cell fails and it has no "b" cell below it, stop and send me the full output.**
Cells 3–6 rebuild the Part A pilot corpus and runs; this is deterministic, and Cell 5 checks it against Part A.
Everything here is **pilot only — never reported as results.**

In [ ]:
# Cell 1 — hardware + disk (models go to /tmp/models: need ~15 GB free there for the judge)
!nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv
!df -h / /tmp /kaggle/working
!free -g

In [ ]:
# Cell 2 — code, retrieval environment, tests
%cd /kaggle/working
!git clone https://github.com/adeelcheena791/rag-embed-transfer.git rag
%cd /kaggle/working/rag
!git log --oneline -1
!pip install -q -r requirements-retrieval.txt
!python -m src.env_report --session pilot_B_retrieval --out results/raw/env_pilot_B_retrieval.json
!python -m pytest -q 2>&1 | tail -3

In [ ]:
# Cell 3 — P1 (re-download, pinned) + P3 join (expect 2255 joined)
!python -m src.data.download --config configs/pilot.yaml
!python -m src.data.join_nq --config configs/pilot.yaml

In [ ]:
# Cell 4 — rebuild the 50k pilot corpus (BM25 over the full corpus, ~6 min)
!python -m src.retrieval.bm25 --config configs/pilot.yaml --corpus full --queries order:200
!python -m src.data.subcorpus --config configs/pilot.yaml --size 50000 --n-queries 200 --name pilot

In [ ]:
# Cell 5 — determinism check against Part A (must print OK)
!echo "9d1eb76d5831cbf2c2d295cd7240529d390b8f89b419fab40d832b84db14d4a4  data/processed/subcorpus_pilot_ids.txt" | sha256sum -c

In [ ]:
# Cell 6 — rebuild the R1 / R4 pilot runs (~6 min)
!CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.encode --config configs/pilot.yaml --models R1,R4 --corpus pilot
!python -m src.retrieval.search --config configs/pilot.yaml --model R1 --corpus pilot --queries order:200
!python -m src.retrieval.search --config configs/pilot.yaml --model R4 --corpus pilot --queries order:200

In [ ]:
# Cell 7 — P8a: freeze the retrieval environment BEFORE vLLM replaces torch
!pip freeze > requirements-retrieval.lock && wc -l requirements-retrieval.lock

In [ ]:
# Cell 8 — generation environment (vLLM 0.30.0 -> torch 2.13; ~5-10 min) + P8b lock
!pip install -q -r requirements-generation.txt 2>&1 | tail -3
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())"
!python -m src.env_report --session pilot_B_generation --out results/raw/env_pilot_B_generation.json
!pip freeze > requirements-generation.lock && wc -l requirements-generation.lock

In [ ]:
# Cell 9 — P5: reader Qwen2.5-7B-Instruct-AWQ via vLLM, one T4; 4 conditions x 200 queries (downloads ~5.6 GB)
!CUDA_VISIBLE_DEVICES=0 python -m src.generation.run --config configs/pilot.yaml --conditions R1,R4,CB,GOLD --corpus pilot --n 200 --k 5

In [ ]:
# Cell 9b — ONLY IF Cell 9 failed: same run without CUDA graphs (common T4 workaround). Resumes where it stopped.
# If this also fails: STOP and send me the outputs of Cells 9 and 9b. Do not switch models.
# !CUDA_VISIBLE_DEVICES=0 python -m src.generation.run --config configs/pilot.yaml --conditions R1,R4,CB,GOLD --corpus pilot --n 200 --k 5 --enforce-eager

In [ ]:
# Cell 10 — score answers (EM / F1 / containment; pilot sanity only)
!python -m src.evaluation.answer --config configs/pilot.yaml --prefix pilot

In [ ]:
# Cell 11 — free disk before the judge download
!rm -rf /tmp/models/Qwen__Qwen2.5-7B-Instruct-AWQ && df -h /tmp

In [ ]:
# Cell 12 — P6: judge Mistral-7B-Instruct-v0.3, fp16, tensor-parallel over BOTH T4s; zero-shot; 3 conditions x 100
# (do not set CUDA_VISIBLE_DEVICES here; downloads ~14.5 GB)
!python -m src.evaluation.judge --config configs/pilot.yaml --conditions R1,R4,GOLD --corpus pilot --n 100 --variant zero_shot

In [ ]:
# Cell 12b — ONLY IF Cell 12 failed: without CUDA graphs. If this also fails: STOP and send me both outputs.
# !python -m src.evaluation.judge --config configs/pilot.yaml --conditions R1,R4,GOLD --corpus pilot --n 100 --variant zero_shot --enforce-eager

In [ ]:
# Cell 13 — ONLY IF Cell 12 ended with "-> few_shot" (zero-shot parse failures > 2%): check that F2 few-shot fixes it
# !python -m src.evaluation.judge --config configs/pilot.yaml --conditions R1,R4,GOLD --corpus pilot --n 100 --variant few_shot

In [ ]:
# Cell 14 — P7: discordant-pair rate -> N_gen (pre-registered rule) + GPU-hour projections vs F4 thresholds
!python -m src.analysis.pilot_power --config configs/pilot.yaml

In [ ]:
# Cell 15 — summary: COPY THIS WHOLE OUTPUT INTO THE CHAT
!python -m src.pilot_summary --config configs/pilot.yaml

In [ ]:
# Cell 16 — package (Output panel -> pilot_part_b_results.zip)
!zip -qr /kaggle/working/pilot_part_b_results.zip results logs configs/resolved_revisions.json requirements-retrieval.lock requirements-generation.lock data/processed/nq_join_stats.json data/processed/subcorpus_pilot_ids.txt
!ls -la /kaggle/working/pilot_part_b_results.zip